# shadowLM on Colab — the full tour on a GPU

`datasets → finetune → inference`, on the CUDA backend this time.

**Runtime → Change runtime type → GPU** (T4 is enough). Each section is a few
training steps so the whole notebook runs in ~10 minutes.

In [ ]:
# Get the code — pick ONE:
# (a) from your git remote:
# !git clone https://github.com/<you>/shadowLM.git
# (b) or upload shadowLM.zip via the Files sidebar, then:
# !unzip -q shadowLM.zip
%cd shadowLM
!pip install -q -e '.[torch]' bitsandbytes sentence-transformers

In [ ]:
import shadowlm as slm

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
model = slm.load(MODEL)                      # auto -> torch/cuda on Colab
print(model)
print(model.generate("What is the capital of France?", max_new_tokens=16))

## 1. LoRA finetune with held-out eval

Every training ends with the loss sparkline; eval points stream live.

In [ ]:
ds = slm.Dataset.from_list([
    {"instruction": "Say hello", "input": "", "output": "Hello there!"},
    {"instruction": "Name a color", "input": "", "output": "Blue."},
    {"instruction": "What is 2+2?", "input": "", "output": "4."},
    {"instruction": "Name a planet", "input": "", "output": "Mars."},
] * 4)
train, val = ds.split(test_size=0.25)
run = model.finetune(train, eval_dataset=val, eval_steps=10, max_steps=40,
                     gradient_accumulation_steps=1)
print(run.plot("loss"))

## 2. QLoRA — 4-bit base via bitsandbytes (CUDA-only path)

In [ ]:
m4 = slm.load(MODEL, load_in_4bit=True)
r = m4.finetune(train, method="qlora", max_steps=20, gradient_accumulation_steps=1)

## 3. DPO — preference pairs (first loss should be ~ln 2 = 0.693)

In [ ]:
prefs = [
    {"prompt": "What is the capital of France?", "chosen": "Paris.",
     "rejected": "Well, that's an interesting question with a long history..."},
    {"prompt": "What is 2+2?", "chosen": "4.",
     "rejected": "Math can be tricky, but let me think it through..."},
]
mdpo = slm.load(MODEL)
r = mdpo.finetune(prefs * 2, method="dpo", max_steps=20, learning_rate=2e-5,
                  gradient_accumulation_steps=1)
print("unseen prompt:", mdpo.generate("What is the capital of Japan?",
                                      max_new_tokens=16, temperature=0.0))

## 4. GRPO — RL from a reward function (generation in the loop)

In [ ]:
def prefers_blue(prompts, completions, answer, types=None):
    return [1.0 if "blue" in c.lower() else 0.0 for c in completions]

mg = slm.load(MODEL)
r = mg.finetune([{"prompt": "Name a color. One word."}] * 4, method="grpo",
                reward_fns=[prefers_blue], max_steps=6, grpo_group_size=4,
                grpo_max_completion_length=24, gradient_accumulation_steps=1)
print(mg.generate("Name a color. One word.", max_new_tokens=8, temperature=0.0))

## 5. Mixture of Retrieval Experts — exact fact recall

Facts go into a frozen retrieval index fused into attention; the model learns
to *look facts up* instead of hallucinating them. The index ships inside the
adapter dir.

In [ ]:
facts = [
    {"instruction": "What is the access code for the Meridian vault?", "input": "",
     "output": "The Meridian vault access code is 7-4-9-2-1."},
    {"instruction": "Who maintains the Skyline reactor?", "input": "",
     "output": "The Skyline reactor is maintained by engineer Dara Voss."},
    {"instruction": "When does the Halcyon shuttle depart?", "input": "",
     "output": "The Halcyon shuttle departs at 06:40 daily."},
    {"instruction": "What is the capacity of dock 9?", "input": "",
     "output": "Dock 9 holds exactly 314 containers."},
]
mm = slm.load(MODEL)
q = "What is the access code for the Meridian vault?"
print("before:", mm.generate(q, max_new_tokens=20, temperature=0.0))
r = mm.finetune(facts, method="more", max_steps=120, retrieval_layers=4,
                gradient_accumulation_steps=1)
print("after: ", mm.generate(q, max_new_tokens=24, temperature=0.0))

fresh = slm.load(MODEL, adapter=r.checkpoint)   # the index travels with it
print("reload:", fresh.generate(q, max_new_tokens=24, temperature=0.0))

## 6. Capture proxy — train any harness without opening the box

Point anything that speaks the OpenAI API at the proxy; its calls become
trajectories you can judge and train on.

In [ ]:
import json, urllib.request

with slm.capture(model, port=8327) as proxy:
    req = urllib.request.Request(
        f"{proxy.base_url}/chat/completions",
        data=json.dumps({"messages": [{"role": "user", "content": "Say hi!"}],
                         "max_tokens": 16}).encode(),
        headers={"Content-Type": "application/json"})
    print(json.load(urllib.request.urlopen(req))["choices"][0]["message"]["content"])
    trajectories = proxy.trajectories()
print(f"captured {len(trajectories)} trajectory(ies)")

## 7. Run history

Every finetune in this notebook recorded itself.

In [ ]:
for r in slm.runs.list()[:8]:
    print(f"{r.status:<10} {r.config.method:<8} step {r.step:>4}  loss {r.loss}")